### Transformers

I'm going to use DistilBERT model; becuase it's faster, requires less memory and it's most suitable for finetuning on Sentiment140 dataset.

In [1]:
import pandas as pd
import numpy as np
import evaluate

from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer)
from sklearn.model_selection import train_test_split

### Preparing Data

In [2]:
df = pd.read_csv("../data/processed/cleaned_sentiment140.csv")

In [3]:
X = df["Clean_Text"].fillna("").astype(str)
y = df["Label"]

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    shuffle=True,
    stratify=y
)

### Hugging Face Dataset

In [5]:
train_dataset = Dataset.from_dict({
    "text": X_train.tolist(),
    "label": y_train.tolist()
})

test_dataset = Dataset.from_dict({
    "text": X_test.tolist(),
    "label": y_test.tolist()
})

### Tokenizer

In [6]:
checkpoint = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(checkpoint)

### Tokenization Function

In [7]:
def tokenizer(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=64
    )

In [8]:
train_dataset = train_dataset.map(tokenizer, batched=True)
test_dataset = test_dataset.map(tokenizer, batched=True)

Map:   0%|          | 0/1279999 [00:01<?, ? examples/s]

TypeError: tokenizer() got an unexpected keyword argument 'truncation'

In [ ]:
train_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

test_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

### Loading Model

In [ ]:
id2label = {
    0: "negative",
    1: "positive"
}

label2id = {
    "negative": 0,
    "positive": 1
}

model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### Metrics

In [ ]:
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

### Metrics Computing

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_metric.compute(
        predictions=predictions,
        references=labels
    )

    f1 = f1_metric.compute(
        predictions=predictions,
        references=labels
    )

    return {
        "accuracy": accuracy["accuracy"],
        "f1": f1["f1"]
    }

### Training Arguments

In [ ]:
training_args = TrainingArguments(
    output_dir="../models/distilbert",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=100,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=True
)

### Trainer API

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

In [ ]:
trainer.evaluate()

In [ ]:
predictions = trainer.predict(test_dataset)

### Saning Model

In [ ]:
model.save_pretrained("../models/transformer_model.pkl")
tokenizer.save_pretrained("../models/transformer_tokenizer.pkl")

### Loading Model

In [ ]:
#from transformers import AutoTokenizer, AutoModelForSequenceClassification
#model = AutoModelForSequenceClassification.from_pretrained("output/model")
#tokenizer = AutoTokenizer.from_pretrained("output/model")